# Обнаружение аномалий в финансовых транзакциях

**Дипломная работа:** Development of an ML Model for Anomaly Detection in Financial Transactions

Пайплайн включает:
- Два датасета: **Credit Card** (реальные данные) и **PaySim** (симуляция мобильных платежей)
- 7 unsupervised моделей для каждого датасета
- Полное сравнение результатов с визуализациями

---

## 1. Импорты и настройки

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.base import BaseEstimator
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope
from sklearn.cluster import KMeans
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    precision_recall_curve
)
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Фиксируем seed везде, чтобы результаты воспроизводились
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("Всё импортировано, seed зафиксирован.")

## 2. Вспомогательные классы и функции

Определяем один раз — используем для обоих датасетов.

In [ ]:
class KMeansAnomalyDetector(BaseEstimator):
    """K-Means как детектор аномалий: чем дальше точка от ближайшего центра кластера, тем подозрительнее."""

    def __init__(self, n_clusters=8, contamination=0.01, random_state=42, n_init=10):
        self.n_clusters = n_clusters
        self.contamination = contamination
        self.random_state = random_state
        self.n_init = n_init

    def fit(self, X, y=None):
        self.kmeans_ = KMeans(
            n_clusters=self.n_clusters,
            random_state=self.random_state,
            n_init=self.n_init
        )
        self.kmeans_.fit(X)
        train_distances = self._distance_to_center(X)
        self.threshold_ = np.quantile(train_distances, 1 - self.contamination)
        return self

    def _distance_to_center(self, X):
        centers = self.kmeans_.cluster_centers_
        labels = self.kmeans_.predict(X)
        return np.linalg.norm(X - centers[labels], axis=1)

    def predict(self, X):
        distances = self._distance_to_center(X)
        return np.where(distances > self.threshold_, -1, 1)

    def decision_function(self, X):
        distances = self._distance_to_center(X)
        return self.threshold_ - distances

    def score_samples(self, X):
        return -self._distance_to_center(X)


class AutoencoderAnomalyDetector(BaseEstimator):
    """Автоэнкодер учится восстанавливать нормальные транзакции.
    Если ошибка восстановления большая — значит транзакция аномальная."""

    def __init__(self, input_dim=None, encoding_dim=14, contamination=0.01,
                 epochs=100, batch_size=2048, random_state=42):
        self.input_dim = input_dim
        self.encoding_dim = encoding_dim
        self.contamination = contamination
        self.epochs = epochs
        self.batch_size = batch_size
        self.random_state = random_state

    def _build_model(self):
        inputs = Input(shape=(self.input_dim,))

        # Энкодер — сжимает данные
        x = Dense(64, activation="relu")(inputs)
        x = BatchNormalization()(x)
        x = Dropout(0.3)(x)
        x = Dense(32, activation="relu")(x)
        x = BatchNormalization()(x)
        x = Dropout(0.2)(x)
        encoded = Dense(self.encoding_dim, activation="relu")(x)

        # Декодер — восстанавливает из сжатого представления
        x = Dense(32, activation="relu")(encoded)
        x = BatchNormalization()(x)
        x = Dropout(0.2)(x)
        x = Dense(64, activation="relu")(x)
        x = BatchNormalization()(x)
        outputs = Dense(self.input_dim, activation="linear")(x)

        model = Model(inputs=inputs, outputs=outputs)
        model.compile(optimizer="adam", loss="mse")
        return model

    def fit(self, X, y=None):
        self.input_dim = X.shape[1] if self.input_dim is None else self.input_dim
        self.model_ = self._build_model()

        # Останавливаемся когда val_loss перестаёт падать
        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )

        # Уменьшаем learning rate если модель застряла
        reduce_lr = ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-6
        )

        self.history_ = self.model_.fit(
            X, X,
            validation_split=0.1,
            epochs=self.epochs,
            batch_size=self.batch_size,
            shuffle=True,
            verbose=0,
            callbacks=[early_stop, reduce_lr]
        )

        # Порог = квантиль ошибки на обучающих данных
        train_recon = self.model_.predict(X, verbose=0)
        train_errors = np.mean(np.square(X - train_recon), axis=1)
        self.threshold_ = np.quantile(train_errors, 1 - self.contamination)
        return self

    def _reconstruction_error(self, X):
        recon = self.model_.predict(X, verbose=0)
        return np.mean(np.square(X - recon), axis=1)

    def predict(self, X):
        errors = self._reconstruction_error(X)
        return np.where(errors > self.threshold_, -1, 1)

    def decision_function(self, X):
        errors = self._reconstruction_error(X)
        return self.threshold_ - errors

    def score_samples(self, X):
        return -self._reconstruction_error(X)

In [ ]:
def evaluate_model(model, model_name, X_train, X_test, y_test, dataset_name):
    """Обучает модель, делает предсказания и считает все метрики."""

    start_train = time.time()
    model.fit(X_train)
    train_time = time.time() - start_train

    start_test = time.time()
    preds_raw = model.predict(X_test)
    test_time = time.time() - start_test

    # sklearn модели возвращают -1 для аномалий, нам нужно 1
    preds = np.where(preds_raw == -1, 1, 0)

    # Скор аномальности: чем выше — тем подозрительнее
    if model_name == "Local Outlier Factor" and hasattr(model, "score_samples"):
        anomaly_scores = -model.score_samples(X_test)
    elif hasattr(model, "decision_function"):
        anomaly_scores = -model.decision_function(X_test)
    elif hasattr(model, "score_samples"):
        anomaly_scores = -model.score_samples(X_test)
    else:
        anomaly_scores = preds.astype(float)

    precision = precision_score(y_test, preds, zero_division=0)
    recall = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    roc_auc = roc_auc_score(y_test, anomaly_scores)
    pr_auc = average_precision_score(y_test, anomaly_scores)

    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()

    return {
        "Dataset": dataset_name,
        "Model": model_name,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Train Time (s)": round(train_time, 2),
        "Test Time (s)": round(test_time, 2)
    }, preds, anomaly_scores


def plot_confusion_matrices(y_test, predictions_dict, dataset_name):
    """Рисует confusion matrix для каждой модели."""
    n = len(predictions_dict)
    cols = 3
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = axes.flatten()

    for i, (name, preds) in enumerate(predictions_dict.items()):
        cm = confusion_matrix(y_test, preds)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        disp.plot(ax=axes[i], colorbar=False)
        axes[i].set_title(name, fontsize=10)

    # Прячем пустые графики
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f"Confusion Matrices — {dataset_name}", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


def plot_roc_curves(y_test, scores_dict, dataset_name):
    """ROC-кривые всех моделей на одном графике."""
    plt.figure(figsize=(10, 7))
    for name, scores in scores_dict.items():
        fpr, tpr, _ = roc_curve(y_test, scores)
        auc_val = roc_auc_score(y_test, scores)
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.3f})")

    plt.plot([0, 1], [0, 1], "k--", alpha=0.3)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves — {dataset_name}")
    plt.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    plt.show()


def plot_pr_curves(y_test, scores_dict, dataset_name):
    """Precision-Recall кривые — более честная метрика при сильном дисбалансе."""
    plt.figure(figsize=(10, 7))
    for name, scores in scores_dict.items():
        prec, rec, _ = precision_recall_curve(y_test, scores)
        ap = average_precision_score(y_test, scores)
        plt.plot(rec, prec, label=f"{name} (AP={ap:.3f})")

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision-Recall Curves — {dataset_name}")
    plt.legend(loc="upper right", fontsize=9)
    plt.tight_layout()
    plt.show()


def run_pipeline(X_train_scaled, X_test_scaled, y_test, contamination, dataset_name):
    """Запускает все 7 моделей, собирает метрики, рисует графики."""

    models = {
        "Isolation Forest": IsolationForest(
            n_estimators=100,
            contamination=contamination,
            random_state=RANDOM_STATE
        ),
        "Tuned Isolation Forest": IsolationForest(
            n_estimators=300,
            max_samples=0.8,
            contamination=contamination,
            random_state=RANDOM_STATE
        ),
        "One-Class SVM": OneClassSVM(
            kernel="rbf",
            nu=contamination,
            gamma="scale"
        ),
        "Local Outlier Factor": LocalOutlierFactor(
            n_neighbors=35,
            contamination=contamination,
            novelty=True
        ),
        "Elliptic Envelope": EllipticEnvelope(
            contamination=contamination,
            random_state=RANDOM_STATE,
            support_fraction=0.999
        ),
        "K-Means": KMeansAnomalyDetector(
            n_clusters=8,
            contamination=contamination,
            random_state=RANDOM_STATE
        ),
        "Autoencoder": AutoencoderAnomalyDetector(
            input_dim=X_train_scaled.shape[1],
            encoding_dim=14,
            contamination=contamination,
            epochs=100,
            batch_size=2048,
            random_state=RANDOM_STATE
        )
    }

    results = []
    predictions_dict = {}
    scores_dict = {}

    for model_name, model in models.items():
        print(f"  Обучаю {model_name}...")
        result, preds, scores = evaluate_model(
            model, model_name,
            X_train_scaled, X_test_scaled, y_test,
            dataset_name
        )
        results.append(result)
        predictions_dict[model_name] = preds
        scores_dict[model_name] = scores
        print(f"    F1={result['F1-score']:.4f}  Precision={result['Precision']:.4f}  Recall={result['Recall']:.4f}")

    results_df = pd.DataFrame(results)

    # Confusion matrices
    plot_confusion_matrices(y_test, predictions_dict, dataset_name)

    # ROC и PR кривые
    plot_roc_curves(y_test, scores_dict, dataset_name)
    plot_pr_curves(y_test, scores_dict, dataset_name)

    return results_df, predictions_dict, scores_dict

---
## 3. Датасет 1: Credit Card Fraud Detection

Реальные транзакции европейских держателей карт (сентябрь 2013).
Признаки V1–V28 — результат PCA-преобразования (анонимизация).
Оригинальные признаки не раскрыты по соображениям конфиденциальности.

### 3.1 Загрузка и очистка

In [ ]:
# Ищем файл в нескольких типичных местах
cc_candidates = [
    "creditcard.csv",
    "/content/creditcard.csv",
    "/content/sample_data/creditcard.csv",
    "/mnt/data/creditcard.csv"
]
cc_path = next((p for p in cc_candidates if os.path.exists(p)), None)
if cc_path is None:
    raise FileNotFoundError("creditcard.csv не найден. Загрузите файл в рабочую директорию.")

cc_df = pd.read_csv(cc_path)
print(f"Загружен из: {cc_path}")
print(f"Размер: {cc_df.shape}")
print(f"Пропуски: {cc_df.isnull().sum().sum()}")
print(f"Дубликаты: {cc_df.duplicated().sum()}")
cc_df.head()

In [ ]:
# Убираем пропуски и дубликаты
cc_df = cc_df.dropna().drop_duplicates().reset_index(drop=True)

print(f"После очистки: {cc_df.shape}")
print(f"\nРаспределение классов:")
print(cc_df["Class"].value_counts())
print(f"\nДоля мошенничества: {cc_df['Class'].mean() * 100:.4f}%")

### 3.2 Разведочный анализ (EDA)

In [ ]:
fraud_cc = cc_df[cc_df["Class"] == 1]
normal_cc = cc_df[cc_df["Class"] == 0]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Распределение суммы: нормальные vs фрод
axes[0].hist(normal_cc["Amount"], bins=50, alpha=0.6, label="Normal", density=True)
axes[0].hist(fraud_cc["Amount"], bins=50, alpha=0.6, label="Fraud", density=True)
axes[0].set_title("Распределение суммы (Amount)")
axes[0].set_xlabel("Amount")
axes[0].legend()

# Время: когда происходят фроды
axes[1].hist(normal_cc["Time"] / 3600, bins=48, alpha=0.6, label="Normal", density=True)
axes[1].hist(fraud_cc["Time"] / 3600, bins=48, alpha=0.6, label="Fraud", density=True)
axes[1].set_title("Распределение по времени (часы)")
axes[1].set_xlabel("Часы от начала")
axes[1].legend()

# Баланс классов
cc_df["Class"].value_counts().plot(kind="bar", ax=axes[2], color=["steelblue", "coral"])
axes[2].set_title("Баланс классов")
axes[2].set_xticklabels(["Normal", "Fraud"], rotation=0)

plt.tight_layout()
plt.show()

### 3.3 Feature engineering и preprocessing

In [ ]:
# log-трансформация суммы — фрод часто на экстремальных суммах (очень маленькие или большие)
cc_df["Amount_log"] = np.log1p(cc_df["Amount"])

# Час суток (циклический) — фрод чаще ночью
cc_df["Hour"] = (cc_df["Time"] % 86400) / 3600
cc_df["Hour_sin"] = np.sin(2 * np.pi * cc_df["Hour"] / 24)
cc_df["Hour_cos"] = np.cos(2 * np.pi * cc_df["Hour"] / 24)

# Убираем оригинальные Time, Amount, Hour (оставляем трансформированные)
cc_features = cc_df.drop(columns=["Class", "Time", "Amount", "Hour"])

X_cc = cc_features
y_cc = cc_df["Class"]

print(f"Признаков: {X_cc.shape[1]}")
print(f"Новые признаки: Amount_log, Hour_sin, Hour_cos")
X_cc.head()

In [ ]:
# Разделяем: обучаемся только на нормальных, тестируем на смеси
X_cc_normal = X_cc[y_cc == 0]
X_cc_fraud = X_cc[y_cc == 1]

X_train_cc, X_test_normal_cc = train_test_split(
    X_cc_normal, test_size=0.2, random_state=RANDOM_STATE
)

# Тестовый набор = нормальные (20%) + ВСЕ мошеннические
X_test_cc = pd.concat([X_test_normal_cc, X_cc_fraud], axis=0)
y_test_cc = np.concatenate([
    np.zeros(len(X_test_normal_cc)),
    np.ones(len(X_cc_fraud))
])

# Масштабируем — fit только на train, transform на test
scaler_cc = StandardScaler()
X_train_cc_scaled = scaler_cc.fit_transform(X_train_cc)
X_test_cc_scaled = scaler_cc.transform(X_test_cc)

# Contamination — доля фрода в данных
cc_contamination = float(np.clip(y_cc.mean(), 1e-4, 0.05))

print(f"Train (только нормальные): {X_train_cc_scaled.shape}")
print(f"Test (нормальные + фроды): {X_test_cc_scaled.shape}")
print(f"Фродов в тесте: {int(y_test_cc.sum())}")
print(f"Contamination: {cc_contamination:.6f}")

### 3.4 Таблица гиперпараметров

In [ ]:
cc_hyperparams = pd.DataFrame([
    {"Model": "Isolation Forest", "Hyperparameters": f"n_estimators=100, contamination={cc_contamination:.6f}"},
    {"Model": "Tuned Isolation Forest", "Hyperparameters": f"n_estimators=300, max_samples=0.8, contamination={cc_contamination:.6f}"},
    {"Model": "One-Class SVM", "Hyperparameters": f"kernel=rbf, nu={cc_contamination:.6f}, gamma=scale"},
    {"Model": "Local Outlier Factor", "Hyperparameters": f"n_neighbors=35, contamination={cc_contamination:.6f}, novelty=True"},
    {"Model": "Elliptic Envelope", "Hyperparameters": f"contamination={cc_contamination:.6f}, support_fraction=0.999"},
    {"Model": "K-Means", "Hyperparameters": f"n_clusters=8, contamination={cc_contamination:.6f}"},
    {"Model": "Autoencoder", "Hyperparameters": f"arch=64-32-14-32-64, epochs=100, batch=2048, dropout=0.2-0.3"}
])
cc_hyperparams

### 3.5 Обучение и оценка 7 моделей

In [ ]:
print("=" * 60)
print("CREDIT CARD — запуск 7 моделей")
print("=" * 60)

cc_results_df, cc_preds, cc_scores = run_pipeline(
    X_train_cc_scaled, X_test_cc_scaled, y_test_cc,
    cc_contamination, "Credit Card"
)

In [ ]:
# Итоговая таблица, отсортированная по F1
cc_results_sorted = cc_results_df.sort_values("F1-score", ascending=False).reset_index(drop=True)
cc_results_sorted.insert(0, "Rank", range(1, len(cc_results_sorted) + 1))
cc_results_sorted[["Rank", "Model", "Precision", "Recall", "F1-score", "ROC-AUC", "PR-AUC", "TP", "FP", "FN", "TN"]].round(4)

### 3.6 Анализ ошибок — какие фроды пропускает лучшая модель?

In [ ]:
# Берём лучшую модель по F1
best_cc_model = cc_results_sorted.iloc[0]["Model"]
best_cc_preds = cc_preds[best_cc_model]

# Индексы фродов в тесте
fraud_mask = y_test_cc == 1
fraud_preds = best_cc_preds[fraud_mask]

caught = fraud_preds.sum()
missed = len(fraud_preds) - caught

print(f"Лучшая модель: {best_cc_model}")
print(f"Пойманных фродов: {int(caught)} из {len(fraud_preds)}")
print(f"Пропущенных фродов: {int(missed)}")

# Сравниваем суммы пойманных и пропущенных
X_test_cc_reset = X_test_cc.reset_index(drop=True)
fraud_amounts = X_test_cc_reset.loc[fraud_mask, "Amount_log"]
caught_amounts = fraud_amounts[fraud_preds == 1]
missed_amounts = fraud_amounts[fraud_preds == 0]

if len(caught_amounts) > 0 and len(missed_amounts) > 0:
    print(f"\nСредняя log(Amount) пойманных: {caught_amounts.mean():.2f}")
    print(f"Средняя log(Amount) пропущенных: {missed_amounts.mean():.2f}")

    plt.figure(figsize=(8, 4))
    plt.hist(caught_amounts, bins=20, alpha=0.6, label="Пойманные")
    plt.hist(missed_amounts, bins=20, alpha=0.6, label="Пропущенные")
    plt.title(f"Суммы пойманных vs пропущенных фродов ({best_cc_model})")
    plt.xlabel("log(Amount)")
    plt.legend()
    plt.tight_layout()
    plt.show()

---
## 4. Датасет 2: PaySim (симуляция мобильных платежей)

Синтетический датасет, имитирующий транзакции мобильных денег.
Важная особенность — здесь есть человекочитаемые признаки: тип транзакции, балансы, суммы.

### 4.1 Загрузка и очистка

In [ ]:
# PaySim — большой файл (6.3M строк), ищем его
ps_candidates = [
    "PS_20174392719_1491204439457_log.csv",
    "paysim.csv",
    "/content/PS_20174392719_1491204439457_log.csv",
    "/content/paysim.csv",
    "/mnt/data/PS_20174392719_1491204439457_log.csv"
]
ps_path = next((p for p in ps_candidates if os.path.exists(p)), None)
if ps_path is None:
    raise FileNotFoundError("PaySim CSV не найден. Загрузите файл.")

ps_df = pd.read_csv(ps_path)
print(f"Загружен из: {ps_path}")
print(f"Размер: {ps_df.shape}")
print(f"Пропуски: {ps_df.isnull().sum().sum()}")
ps_df.head()

In [ ]:
# Чистим данные
ps_df = ps_df.dropna().drop_duplicates().reset_index(drop=True)

# Переименовываем целевую колонку для единообразия
ps_df = ps_df.rename(columns={"isFraud": "Class"})

print(f"После очистки: {ps_df.shape}")
print(f"\nРаспределение классов:")
print(ps_df["Class"].value_counts())
print(f"\nДоля мошенничества: {ps_df['Class'].mean() * 100:.4f}%")
print(f"\nТипы транзакций:")
print(ps_df["type"].value_counts())

### 4.2 Разведочный анализ (EDA)

In [ ]:
fraud_ps = ps_df[ps_df["Class"] == 1]
normal_ps = ps_df[ps_df["Class"] == 0]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# В каких типах транзакций прячется фрод?
fraud_by_type = fraud_ps["type"].value_counts()
fraud_by_type.plot(kind="bar", ax=axes[0], color="coral")
axes[0].set_title("Фрод по типу транзакции")
axes[0].set_ylabel("Количество")

# Распределение сумм
axes[1].hist(np.log1p(normal_ps["amount"]), bins=50, alpha=0.6, label="Normal", density=True)
axes[1].hist(np.log1p(fraud_ps["amount"]), bins=50, alpha=0.6, label="Fraud", density=True)
axes[1].set_title("log(amount) — Normal vs Fraud")
axes[1].legend()

# Баланс классов
ps_df["Class"].value_counts().plot(kind="bar", ax=axes[2], color=["steelblue", "coral"])
axes[2].set_title("Баланс классов")
axes[2].set_xticklabels(["Normal", "Fraud"], rotation=0)

plt.tight_layout()
plt.show()

### 4.3 Feature engineering

В PaySim ключевой сигнал — это **несовпадение** суммы транзакции с реальным изменением баланса.
Мошенники переводят деньги, но балансы обновляются некорректно.

In [ ]:
# Сколько реально ушло со счёта отправителя
ps_df["balance_diff_orig"] = ps_df["oldbalanceOrg"] - ps_df["newbalanceOrig"]

# Сколько реально пришло на счёт получателя
ps_df["balance_diff_dest"] = ps_df["newbalanceDest"] - ps_df["oldbalanceDest"]

# Ошибка: разница между заявленной суммой и реальным списанием
# Если error != 0, значит что-то нечисто
ps_df["error_orig"] = ps_df["balance_diff_orig"] - ps_df["amount"]
ps_df["error_dest"] = ps_df["balance_diff_dest"] - ps_df["amount"]

# Логарифм суммы
ps_df["amount_log"] = np.log1p(ps_df["amount"])

# Кодируем тип транзакции числом
ps_df["type_code"] = ps_df["type"].astype("category").cat.codes

# Убираем текстовые столбцы и ненужные
ps_features = ps_df.drop(columns=[
    "Class", "type", "nameOrig", "nameDest", "isFlaggedFraud",
    "amount"  # оставляем amount_log вместо сырого amount
])

X_ps = ps_features
y_ps = ps_df["Class"]

print(f"Признаков: {X_ps.shape[1]}")
print(f"Новые признаки: balance_diff_orig, balance_diff_dest, error_orig, error_dest, amount_log")
X_ps.head()

### 4.4 Preprocessing

In [ ]:
X_ps_normal = X_ps[y_ps == 0]
X_ps_fraud = X_ps[y_ps == 1]

X_train_ps, X_test_normal_ps = train_test_split(
    X_ps_normal, test_size=0.2, random_state=RANDOM_STATE
)

X_test_ps = pd.concat([X_test_normal_ps, X_ps_fraud], axis=0)
y_test_ps = np.concatenate([
    np.zeros(len(X_test_normal_ps)),
    np.ones(len(X_ps_fraud))
])

scaler_ps = StandardScaler()
X_train_ps_scaled = scaler_ps.fit_transform(X_train_ps)
X_test_ps_scaled = scaler_ps.transform(X_test_ps)

ps_contamination = float(np.clip(y_ps.mean(), 1e-4, 0.05))

print(f"Train (только нормальные): {X_train_ps_scaled.shape}")
print(f"Test (нормальные + фроды): {X_test_ps_scaled.shape}")
print(f"Фродов в тесте: {int(y_test_ps.sum())}")
print(f"Contamination: {ps_contamination:.6f}")

### 4.5 Обучение и оценка 7 моделей

In [ ]:
print("=" * 60)
print("PAYSIM — запуск 7 моделей")
print("=" * 60)

ps_results_df, ps_preds, ps_scores = run_pipeline(
    X_train_ps_scaled, X_test_ps_scaled, y_test_ps,
    ps_contamination, "PaySim"
)

In [ ]:
ps_results_sorted = ps_results_df.sort_values("F1-score", ascending=False).reset_index(drop=True)
ps_results_sorted.insert(0, "Rank", range(1, len(ps_results_sorted) + 1))
ps_results_sorted[["Rank", "Model", "Precision", "Recall", "F1-score", "ROC-AUC", "PR-AUC", "TP", "FP", "FN", "TN"]].round(4)

---
## 5. Сравнение двух датасетов

### 5.1 Сводная таблица

In [ ]:
# Объединяем результаты
combined_df = pd.concat([cc_results_df, ps_results_df], ignore_index=True)
combined_df = combined_df.sort_values(["Dataset", "F1-score"], ascending=[True, False]).reset_index(drop=True)

combined_df[["Dataset", "Model", "Precision", "Recall", "F1-score", "ROC-AUC", "PR-AUC"]].round(4)

### 5.2 Лучшая модель для каждого датасета

In [ ]:
best_per_dataset = combined_df.loc[
    combined_df.groupby("Dataset")["F1-score"].idxmax()
].reset_index(drop=True)

print("Лучшая модель для каждого датасета:")
best_per_dataset[["Dataset", "Model", "F1-score", "Precision", "Recall", "ROC-AUC", "PR-AUC"]].round(4)

### 5.3 Таблица рангов

In [ ]:
rank_table = combined_df.copy()
rank_table["Rank"] = rank_table.groupby("Dataset")["F1-score"].rank(
    method="dense", ascending=False
).astype(int)

rank_table = rank_table[["Dataset", "Rank", "Model", "F1-score", "ROC-AUC", "PR-AUC"]]
rank_table = rank_table.sort_values(["Dataset", "Rank"]).reset_index(drop=True)
rank_table.round(4)

### 5.4 Визуализация сравнения

In [ ]:
# F1-score: бок о бок для двух датасетов
pivot_f1 = combined_df.pivot(index="Model", columns="Dataset", values="F1-score")
ax = pivot_f1.plot(kind="bar", figsize=(12, 6))
ax.set_title("Сравнение F1-score моделей")
ax.set_ylabel("F1-score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# PR-AUC: бок о бок
pivot_pr = combined_df.pivot(index="Model", columns="Dataset", values="PR-AUC")
ax = pivot_pr.plot(kind="bar", figsize=(12, 6))
ax.set_title("Сравнение PR-AUC моделей")
ax.set_ylabel("PR-AUC")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Precision vs Recall scatter для каждого датасета
for dataset_name in combined_df["Dataset"].unique():
    subset = combined_df[combined_df["Dataset"] == dataset_name]

    plt.figure(figsize=(8, 6))
    plt.scatter(subset["Recall"], subset["Precision"], s=120, zorder=5)

    for _, row in subset.iterrows():
        plt.annotate(row["Model"], (row["Recall"], row["Precision"]),
                     fontsize=9, textcoords="offset points", xytext=(5, 5))

    plt.title(f"Precision vs Recall — {dataset_name}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.xlim(-0.05, 1.05)
    plt.ylim(-0.05, 1.05)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Тепловая карта F1-score
heatmap_data = combined_df.pivot(index="Model", columns="Dataset", values="F1-score")

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax)
ax.set_title("F1-score по моделям и датасетам")
plt.tight_layout()
plt.show()

In [ ]:
# Время обучения
time_data = combined_df.pivot(index="Model", columns="Dataset", values="Train Time (s)")
ax = time_data.plot(kind="bar", figsize=(12, 5))
ax.set_title("Время обучения (секунды)")
ax.set_ylabel("Секунды")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 5.5 Сводная информация о датасетах

In [ ]:
dataset_summary = pd.DataFrame([
    {
        "Dataset": "Credit Card",
        "Total Samples": len(cc_df),
        "Normal": int((cc_df["Class"] == 0).sum()),
        "Fraud": int((cc_df["Class"] == 1).sum()),
        "Fraud %": round(cc_df["Class"].mean() * 100, 4),
        "Features (original)": 31,
        "Features (engineered)": X_cc.shape[1],
        "Source": "Реальные данные (Kaggle, ULB)"
    },
    {
        "Dataset": "PaySim",
        "Total Samples": len(ps_df),
        "Normal": int((ps_df["Class"] == 0).sum()),
        "Fraud": int((ps_df["Class"] == 1).sum()),
        "Fraud %": round(ps_df["Class"].mean() * 100, 4),
        "Features (original)": 11,
        "Features (engineered)": X_ps.shape[1],
        "Source": "Синтетическая симуляция"
    }
])
dataset_summary

## 6. Экспорт результатов

In [ ]:
# Сохраняем все таблицы в CSV
cc_results_df.to_csv("creditcard_results.csv", index=False)
ps_results_df.to_csv("paysim_results.csv", index=False)
combined_df.to_csv("combined_results.csv", index=False)
rank_table.to_csv("model_ranking.csv", index=False)
best_per_dataset.to_csv("best_models.csv", index=False)
dataset_summary.to_csv("dataset_summary.csv", index=False)

# Всё в один Excel файл (удобно для диплома)
with pd.ExcelWriter("diploma_results.xlsx", engine="openpyxl") as writer:
    dataset_summary.to_excel(writer, sheet_name="Datasets", index=False)
    cc_results_sorted.round(4).to_excel(writer, sheet_name="CreditCard", index=False)
    ps_results_sorted.round(4).to_excel(writer, sheet_name="PaySim", index=False)
    combined_df.round(4).to_excel(writer, sheet_name="Combined", index=False)
    rank_table.round(4).to_excel(writer, sheet_name="Ranking", index=False)
    best_per_dataset.round(4).to_excel(writer, sheet_name="BestModels", index=False)

print("Все результаты сохранены.")
print("Файлы: creditcard_results.csv, paysim_results.csv, combined_results.csv,")
print("        model_ranking.csv, best_models.csv, dataset_summary.csv,")
print("        diploma_results.xlsx")

## 7. Выводы

Автоматически сгенерированные выводы на основе результатов:

In [ ]:
print("=" * 60)
print("ВЫВОДЫ")
print("=" * 60)

for _, row in best_per_dataset.iterrows():
    print(f"\n{row['Dataset']}:")
    print(f"  Лучшая модель: {row['Model']}")
    print(f"  F1-score:  {row['F1-score']:.4f}")
    print(f"  Precision: {row['Precision']:.4f}")
    print(f"  Recall:    {row['Recall']:.4f}")
    print(f"  PR-AUC:    {row['PR-AUC']:.4f}")

print("\n" + "=" * 60)

# Какая модель стабильнее всего на обоих датасетах?
avg_f1 = combined_df.groupby("Model")["F1-score"].mean().sort_values(ascending=False)
print(f"\nСамая стабильная модель (средний F1 по двум датасетам):")
print(f"  {avg_f1.index[0]} — средний F1 = {avg_f1.iloc[0]:.4f}")

print(f"\n\nРанжирование по среднему F1:")
for i, (model, score) in enumerate(avg_f1.items(), 1):
    print(f"  {i}. {model}: {score:.4f}")